# MO-WOA evaluation - comparison with other algorithms

Thesis: Multi-Objective Whale Optimization Algorithm (MO-WOA) for extractive text summarization.

Compares MO-WOA with 5 baseline/comparison algorithms on the same 150 IndoSum articles,
using the same metrics and evaluation config.

## Algorithms
| # | Algorithm | Type | Implementation |
|---|-----------|------|----------------|
| 1 | LEAD-3 | Heuristic baseline | First 3 sentences |
| 2 | TextRank | Graph-based (TF-IDF) | TF-IDF + PageRank |
| 3 | LexRank | Graph-based (TF-IDF) | TF-IDF + cosine threshold + PageRank |
| 4 | WOA (Single-obj) | Metaheuristic | Scalarized fitness, no Pareto dominance |
| 5 | IndoBERT | Deep learning | Fine-tuned extractive summarization |
| 6 | MO-WOA | Proposed | Pareto dominance + crowding distance + knee-point |

## Metrics
- ROUGE-1, ROUGE-2, ROUGE-L (P, R, F1)
- compression ratio (summary length / source length)
- n-gram redundancy (trigram, threshold 0.3)

## Config
- `summary_size = 3` sentences for every algorithm
- `use_stemmer = False`, the english stemmer doesn't work for Indonesian
- IndoSum `train.01.jsonl`, first 150 articles

## Data
`golden_summaries` and `source_texts` are loaded from `mowoa_summaries_150.json`,
the same references MO-WOA was evaluated against, so every algorithm is compared on the same data.

## 1. Environment Setup

In [ ]:
!pip install rouge-score kagglehub scikit-learn networkx -q
!pip install sentence-transformers -q
!pip install transformers torch accelerate -q

import numpy as np
import pandas as pd
import json
import re
import os
import time
import warnings
from collections import Counter
from typing import List, Dict, Tuple

from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity
import networkx as nx

warnings.filterwarnings('ignore')
np.random.seed(42)

print("imports ok")
SUMMARY_SIZE = 3  # same for every algorithm
N_ARTICLES = 150
print(f"{N_ARTICLES} articles, {SUMMARY_SIZE} sentences per summary")

In [ ]:
# mount drive so the cache survives colab restarts
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CACHE_DIR = '/content/drive/MyDrive/thesis_mowoa_cache'
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)

print(f"drive mounted: {os.path.exists('/content/drive/MyDrive')}")
print(f"cache dir: {DRIVE_CACHE_DIR} (exists: {os.path.exists(DRIVE_CACHE_DIR)})")

existing = os.listdir(DRIVE_CACHE_DIR) if os.path.exists(DRIVE_CACHE_DIR) else []
if existing:
    print("\ncached files:")
    for f in existing:
        path = os.path.join(DRIVE_CACHE_DIR, f)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {f} ({size_mb:.1f} MB)")
else:
    print("\ncache dir is empty (first run)")

## 2. Shared Utilities

All algorithms share the same sentence extraction, evaluation metrics, and output format
to ensure a fair and consistent comparison.

In [ ]:
def split_text_into_sentences(paragraphs):
    """IndoSum paragraphs are lists of sentences of tokens, join them back but keep the original sentence boundaries."""
    sentences = []
    for paragraph in paragraphs:
        for sentence_tokens in paragraph:
            sentence = ' '.join(sentence_tokens)
            sentence = sentence.replace(' ,', ',').replace(' .', '.')
            sentence = sentence.replace(' :', ':').replace(' ;', ';')
            sentence = sentence.replace(' !', '!').replace(' ?', '?')
            sentence = sentence.replace(' )', ')').replace('( ', '(')
            if sentence.strip():
                sentences.append(sentence.strip())
    return sentences


def tokens_to_narrative(tokenized_sentences):
    """Tokenized reference summary -> one string."""
    sentences = []
    for tokens in tokenized_sentences:
        sentence = " ".join(tokens)
        sentence = sentence.replace(" ,", ",").replace(" .", ".")
        sentence = sentence.replace(" :", ":").replace(" ;", ";")
        sentence = sentence.replace(" !", "!").replace(" ?", "?")
        sentences.append(sentence)
    return " ".join(sentences)


def reconstruct_source_text(paragraphs):
    """Full article text, used for the compression ratio."""
    sentences = split_text_into_sentences(paragraphs)
    return ' '.join(sentences)


def compute_ngram_redundancy(summary: str, n: int = 3, threshold: float = 0.3) -> float:
    """max n-gram freq / total n-grams, minus threshold. 0 = no redundancy."""
    words = re.findall(r'\b\w+\b', summary.lower())
    if len(words) < n:
        return 0.0

    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    ngram_counts = Counter(ngrams)
    total_ngrams = len(ngrams)

    if total_ngrams == 0:
        return 0.0

    max_freq = max(ngram_counts.values()) if ngram_counts else 0
    redundancy = max_freq / total_ngrams

    return max(0.0, redundancy - threshold)


def compute_ngram_redundancy_batch(summaries: List[str], n: int = 3,
                                   threshold: float = 0.3) -> dict:
    """n-gram redundancy for a list of summaries."""
    scores = [compute_ngram_redundancy(s, n=n, threshold=threshold) for s in summaries]
    return {
        'avg_redundancy': np.mean(scores),
        'redundancy_scores': np.array(scores),
        'n_gram': n,
        'threshold': threshold,
    }


def compute_compression_ratio(source_texts: List[str],
                               summary_texts: List[str]) -> dict:
    """summary length / source length in chars, lower = more compressed."""
    source_lengths = np.array([len(t) for t in source_texts])
    summary_lengths = np.array([len(t) for t in summary_texts])
    ratios = summary_lengths / np.maximum(source_lengths, 1)

    return {
        'avg_compression_ratio': np.mean(ratios),
        'compression_ratios': ratios,
        'avg_source_length': np.mean(source_lengths),
        'avg_summary_length': np.mean(summary_lengths),
    }


def evaluate_summaries(generated_summaries: List[str],
                       golden_summaries: List[str],
                       source_texts: List[str],
                       algorithm_name: str) -> dict:
    """ROUGE-1/2/L (P, R, F1), compression ratio and trigram redundancy, same for every algorithm."""
    assert len(generated_summaries) == len(golden_summaries) == len(source_texts), \
        f"Length mismatch: gen={len(generated_summaries)}, ref={len(golden_summaries)}, src={len(source_texts)}"

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

    rouge1_p, rouge1_r, rouge1_f = [], [], []
    rouge2_p, rouge2_r, rouge2_f = [], [], []
    rougeL_p, rougeL_r, rougeL_f = [], [], []

    for gen, ref in zip(generated_summaries, golden_summaries):
        scores = scorer.score(ref, gen)
        rouge1_p.append(scores['rouge1'].precision)
        rouge1_r.append(scores['rouge1'].recall)
        rouge1_f.append(scores['rouge1'].fmeasure)
        rouge2_p.append(scores['rouge2'].precision)
        rouge2_r.append(scores['rouge2'].recall)
        rouge2_f.append(scores['rouge2'].fmeasure)
        rougeL_p.append(scores['rougeL'].precision)
        rougeL_r.append(scores['rougeL'].recall)
        rougeL_f.append(scores['rougeL'].fmeasure)

    cr = compute_compression_ratio(source_texts, generated_summaries)

    nr = compute_ngram_redundancy_batch(generated_summaries)

    results = {
        'algorithm': algorithm_name,
        'rouge1_precision': np.mean(rouge1_p),
        'rouge1_recall': np.mean(rouge1_r),
        'rouge1_f1': np.mean(rouge1_f),
        'rouge2_precision': np.mean(rouge2_p),
        'rouge2_recall': np.mean(rouge2_r),
        'rouge2_f1': np.mean(rouge2_f),
        'rougeL_precision': np.mean(rougeL_p),
        'rougeL_recall': np.mean(rougeL_r),
        'rougeL_f1': np.mean(rougeL_f),
        'compression_ratio': cr['avg_compression_ratio'],
        'avg_summary_length': cr['avg_summary_length'],
        'ngram_redundancy': nr['avg_redundancy'],
        # per article, for the significance tests
        '_rouge1_f1_per_article': np.array(rouge1_f),
        '_rouge2_f1_per_article': np.array(rouge2_f),
        '_rougeL_f1_per_article': np.array(rougeL_f),
        '_compression_ratios': cr['compression_ratios'],
        '_redundancy_scores': nr['redundancy_scores'],
    }

    print(f"\n{algorithm_name} ({len(generated_summaries)} articles)")
    print(f"  ROUGE-1  P={results['rouge1_precision']:.4f}  R={results['rouge1_recall']:.4f}  F1={results['rouge1_f1']:.4f}")
    print(f"  ROUGE-2  P={results['rouge2_precision']:.4f}  R={results['rouge2_recall']:.4f}  F1={results['rouge2_f1']:.4f}")
    print(f"  ROUGE-L  P={results['rougeL_precision']:.4f}  R={results['rougeL_recall']:.4f}  F1={results['rougeL_f1']:.4f}")
    print(f"  CR {results['compression_ratio']:.1%}, avg len {results['avg_summary_length']:.0f} chars, "
          f"redundancy {results['ngram_redundancy']:.4f}")

    return results


print("utils loaded")

## 3. Load Canonical Evaluation Data from MO-WOA JSON

`golden_summaries` and `source_texts` come from `mowoa_summaries_150.json`, the same
references MO-WOA was evaluated against in the v5.1 notebook. So all algorithms use exactly
the same reference summaries and there are no small differences from rebuilding the text.

The JSON has:
- `generated_summaries`: 150 MO-WOA summaries (precomputed)
- `golden_summaries`: 150 reference summaries
- `source_texts`: 150 full source articles

In [ ]:
# golden_summaries and source_texts come from this json, every algorithm
# (baselines too) is scored against these references
MOWOA_JSON_PATH = 'mowoa_summaries_150_rc1_w32.json'

with open(MOWOA_JSON_PATH, 'r', encoding='utf-8') as f:
    mowoa_data = json.load(f)

mowoa_summaries_all = mowoa_data['generated_summaries']
golden_summaries_all = mowoa_data['golden_summaries']
source_texts_all = mowoa_data['source_texts']

assert len(mowoa_summaries_all) >= N_ARTICLES, \
    f"JSON contains only {len(mowoa_summaries_all)} entries but N_ARTICLES={N_ARTICLES}"

# N_ARTICLES can be set lower (e.g. 3) for quick tests
mowoa_summaries = mowoa_summaries_all[:N_ARTICLES]
golden_summaries = golden_summaries_all[:N_ARTICLES]
source_texts = source_texts_all[:N_ARTICLES]

print(f"{MOWOA_JSON_PATH}: {len(mowoa_summaries_all)} in json, using first {N_ARTICLES}")
print(f"mowoa {len(mowoa_summaries)}, golden {len(golden_summaries)}, source {len(source_texts)}")
print(f"avg length (chars): golden {np.mean([len(s) for s in golden_summaries]):.0f}, "
      f"source {np.mean([len(s) for s in source_texts]):.0f}, "
      f"mowoa {np.mean([len(s) for s in mowoa_summaries]):.0f}")

print(f"\narticle 0 golden: {golden_summaries[0][:100]}...")
print(f"article 0 mowoa:  {mowoa_summaries[0][:100]}...")

## 4. Load IndoSum Dataset & Derive Sentence Lists

Load the raw IndoSum dataset to get `all_sentences`, the per-article sentence lists
the baselines (TextRank, LexRank, etc.) need to pick their own sentences.

Then a consistency check makes sure the source texts rebuilt from the dataset match
the ones in the canonical JSON, so all algorithms work on the same articles.

In [ ]:
# # =========================================================================
# # CELL 4 — Load dataset, extract sentences, validate consistency
# # =========================================================================
# import kagglehub

# base_path = kagglehub.dataset_download("linkgish/indosum")
# path = f"{base_path}/indosum"

# temp_train_df = pd.read_json(f'{path}/train.01.jsonl', lines=True)
# train_df = temp_train_df[['paragraphs', 'summary']].copy()

# print(f"Loaded {len(train_df)} articles from train.01.jsonl")
# print(f"Using first {N_ARTICLES} articles for evaluation\n")

# # --- Extract sentence lists (needed by baselines to run their algorithms) ---
# all_sentences = []
# for idx in range(N_ARTICLES):
#     sents = split_text_into_sentences(train_df.paragraphs[idx])
#     all_sentences.append(sents)

# print(f"Extracted sentences for {len(all_sentences)} articles")
# print(f"Avg sentences per article: {np.mean([len(s) for s in all_sentences]):.1f}")

# # --- CONSISTENCY VALIDATION ---
# # Verify that source texts derived from the dataset match the canonical JSON.
# # This catches any discrepancy between the v5.1 pipeline and this notebook's
# # sentence extraction, which would invalidate the comparison.

# print("\nValidating consistency between dataset and canonical JSON...")
# mismatches = []
# for idx in range(N_ARTICLES):
#     derived_source = ' '.join(all_sentences[idx])
#     canonical_source = source_texts[idx]
#     if derived_source != canonical_source:
#         mismatches.append(idx)

# if len(mismatches) == 0:
#     print("  PASSED: All 150 derived source texts match the canonical JSON exactly.")
# else:
#     print(f"  WARNING: {len(mismatches)} articles have source text mismatches!")
#     print(f"  Mismatched indices: {mismatches[:10]}{'...' if len(mismatches) > 10 else ''}")
#     # Show first mismatch for debugging
#     idx0 = mismatches[0]
#     derived = ' '.join(all_sentences[idx0])
#     canonical = source_texts[idx0]
#     print(f"\n  Article {idx0} — first 200 chars:")
#     print(f"    Derived:   {derived[:200]}")
#     print(f"    Canonical: {canonical[:200]}")
#     print(f"\n  NOTE: Using canonical JSON source_texts for evaluation regardless.")
#     print(f"  Baselines still operate on the derived all_sentences for sentence selection.")

# # Also validate golden summaries
# golden_mismatches = []
# for idx in range(N_ARTICLES):
#     derived_golden = tokens_to_narrative(train_df.summary[idx])
#     canonical_golden = golden_summaries[idx]
#     if derived_golden != canonical_golden:
#         golden_mismatches.append(idx)

# if len(golden_mismatches) == 0:
#     print("  PASSED: All 150 derived golden summaries match the canonical JSON exactly.")
# else:
#     print(f"  WARNING: {len(golden_mismatches)} golden summaries have mismatches!")
#     print(f"  Mismatched indices: {golden_mismatches[:10]}")
#     idx0 = golden_mismatches[0]
#     print(f"\n  Article {idx0}:")
#     print(f"    Derived:   {tokens_to_narrative(train_df.summary[idx0])[:200]}")
#     print(f"    Canonical: {golden_summaries[idx0][:200]}")
#     print(f"\n  NOTE: Using canonical JSON golden_summaries for ALL evaluations.")

# split the canonical source_texts back into sentences. don't use train_df.paragraphs,
# its article order can differ from the json and the baselines would end up
# summarizing the wrong article

import re

def split_canonical_source(source_text):
    """source_texts were built with ' '.join(sentences), so just split on sentence-ending punctuation."""
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z"\'\(])', source_text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

all_sentences = []
for idx in range(N_ARTICLES):
    sents = split_canonical_source(source_texts[idx])
    all_sentences.append(sents)

print(f"{len(all_sentences)} articles, avg {np.mean([len(s) for s in all_sentences]):.1f} sentences per article")

print("\nalignment check, article 0 (all four should be the same story):")
print(f"  first sentence: {all_sentences[0][0][:120]}")
print(f"  source: {source_texts[0][:120]}")
print(f"  mowoa: {mowoa_summaries[0][:120]}")
print(f"  golden: {golden_summaries[0][:120]}")

## 5. Evaluate MO-WOA (Proposed Method)

MO-WOA summaries were precomputed in the v5.1 notebook and are already loaded
from the JSON. We now evaluate them through the unified evaluation pipeline
to ensure metric computation is identical to all baselines.

In [ ]:
# summaries were already generated in the MO-WOA notebook, just score them here
mowoa_results = evaluate_summaries(
    mowoa_summaries, golden_summaries, source_texts, "MO-WOA"
)

## 6. Baseline: LEAD-3

Select the first 3 sentences of each article. This is the standard
heuristic baseline for news summarization, exploiting the inverted-pyramid
structure of journalistic writing.

In [ ]:
def lead_n_summarize(sentences: List[str], n: int = 3) -> str:
    """First n sentences."""
    selected = sentences[:n]
    return ' '.join(selected)

lead3_summaries = []
for idx in range(N_ARTICLES):
    summary = lead_n_summarize(all_sentences[idx], n=SUMMARY_SIZE)
    lead3_summaries.append(summary)

lead3_results = evaluate_summaries(
    lead3_summaries, golden_summaries, source_texts, "LEAD-3"
)

## 7. Baseline: TextRank

TextRank (Mihalcea & Tarau, 2004) builds a sentence similarity graph from
TF-IDF cosine similarity and runs PageRank to find the most central sentences.

TF-IDF -> cosine similarity -> weighted undirected graph -> PageRank -> top 3 by score.

In [ ]:
def textrank_summarize(sentences: List[str], n: int = 3) -> str:
    """TF-IDF cosine graph -> PageRank -> top n sentences, kept in document order."""
    if len(sentences) <= n:
        return ' '.join(sentences)

    try:
        vectorizer = TfidfVectorizer(max_features=5000)
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        # empty vocab on very short articles
        return ' '.join(sentences[:n])

    sim_matrix = sklearn_cosine_similarity(tfidf_matrix)

    G = nx.Graph()
    for i in range(len(sentences)):
        G.add_node(i)
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if sim_matrix[i, j] > 0:
                G.add_edge(i, j, weight=sim_matrix[i, j])

    if len(G.edges()) == 0:
        # no edges, uniform scores so it ends up as first n
        scores = {i: 1.0 / len(sentences) for i in range(len(sentences))}
    else:
        scores = nx.pagerank(G, weight='weight', alpha=0.85)

    # top n by score, back in document order
    ranked = sorted(scores.keys(), key=lambda i: scores[i], reverse=True)
    selected_indices = sorted(ranked[:n])
    return ' '.join([sentences[i] for i in selected_indices])

textrank_summaries = []
for idx in range(N_ARTICLES):
    summary = textrank_summarize(all_sentences[idx], n=SUMMARY_SIZE)
    textrank_summaries.append(summary)
    if (idx + 1) % 50 == 0:
        print(f"textrank {idx + 1}/{N_ARTICLES}")

textrank_results = evaluate_summaries(
    textrank_summaries, golden_summaries, source_texts, "TextRank"
)

## 8. Baseline: LexRank

LexRank (Erkan & Radev, 2004) is like TextRank but only connects sentence pairs whose
cosine similarity is above a threshold, so weak links are dropped and the graph is sparser.

TF-IDF -> cosine similarity -> threshold 0.1 -> PageRank -> top 3.

This is the standalone LexRank baseline with TF-IDF similarity, not the LexRank scores
used inside MO-WOA (those use BGE-M3 embeddings).

In [ ]:
def lexrank_summarize(sentences: List[str], n: int = 3,
                      threshold: float = 0.1) -> str:
    """Like textrank, but edges with similarity <= threshold are dropped before PageRank."""
    if len(sentences) <= n:
        return ' '.join(sentences)

    try:
        vectorizer = TfidfVectorizer(max_features=5000)
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        return ' '.join(sentences[:n])

    sim_matrix = sklearn_cosine_similarity(tfidf_matrix)

    G = nx.Graph()
    for i in range(len(sentences)):
        G.add_node(i)
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if sim_matrix[i, j] > threshold:
                G.add_edge(i, j, weight=sim_matrix[i, j])

    if len(G.edges()) == 0:
        scores = {i: 1.0 / len(sentences) for i in range(len(sentences))}
    else:
        scores = nx.pagerank(G, weight='weight', alpha=0.85)

    ranked = sorted(scores.keys(), key=lambda i: scores[i], reverse=True)
    selected_indices = sorted(ranked[:n])
    return ' '.join([sentences[i] for i in selected_indices])

lexrank_summaries = []
for idx in range(N_ARTICLES):
    summary = lexrank_summarize(all_sentences[idx], n=SUMMARY_SIZE, threshold=0.1)
    lexrank_summaries.append(summary)
    if (idx + 1) % 50 == 0:
        print(f"lexrank {idx + 1}/{N_ARTICLES}")

lexrank_results = evaluate_summaries(
    lexrank_summaries, golden_summaries, source_texts, "LexRank"
)

## 9. Ablation: Single-Objective WOA

Ablation baseline for MO-WOA. Same WOA setup (binary encoding, sigmoid transfer,
encircling/spiral/exploration, repair operator), but Pareto dominance is replaced by one scalar fitness:

$$\text{fitness} = \text{coverage} - \text{redundancy} - 2 \times \text{length\_penalty}^2$$

Differences from MO-WOA:
- leader = max scalar fitness (not crowding distance on the Pareto front)
- final solution = max scalar fitness (not knee point)
- no Pareto front

Pipeline: BGE-M3 embeddings -> cosine similarity graph -> LexRank -> WOA with scalar fitness.

This cell is slow (~1-3 hours for 150 articles on GPU), it loads BGE-M3 and runs the full pipeline per article.

In [ ]:
# single-objective WOA (ablation), ~1-3h on colab T4.
# set RUN_SINGLE_OBJ_WOA = False to load the saved json instead

RUN_SINGLE_OBJ_WOA = True
WOA_JSON_PATH = f'{DRIVE_CACHE_DIR}/woa_singleobj_summaries_150.json'

if RUN_SINGLE_OBJ_WOA:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim

    # same embedding model as MO-WOA
    print("loading bge-m3")
    bge_model = SentenceTransformer('BAAI/bge-m3')

    def build_similarity_graph(sentences, bge_model, threshold=0.1):
        """Embed sentences with BGE-M3, build cosine similarity graph."""
        embeddings = bge_model.encode(sentences, normalize_embeddings=True,
                                     show_progress_bar=False)
        sim_matrix = cos_sim(embeddings)

        G = nx.Graph()
        for i, s in enumerate(sentences):
            G.add_node(i, text=s.strip(), original_text=s.strip())
        for i in range(len(sentences)):
            for j in range(i + 1, len(sentences)):
                if sim_matrix[i, j] > threshold:
                    G.add_edge(i, j, weight=float(sim_matrix[i, j]))
        return G, sim_matrix

    def compute_lexrank(graph, damping=0.85):
        """PageRank scores on the similarity graph, also stored as node attr 'importance'."""
        if len(graph.edges()) == 0:
            n = len(graph.nodes())
            scores = {i: 1.0 / n for i in range(n)}
        else:
            scores = nx.pagerank(graph, weight='weight', alpha=damping)
        nx.set_node_attributes(graph, scores, 'importance')
        return scores

    def single_objective_woa(graph, pop_size=30, max_iter=50,
                              summary_size=3, position_bonus=None):
        """
        Same WOA as MO-WOA but with one scalar fitness,
        coverage - redundancy - 2 * length_penalty^2, used for both the leader
        and the final pick (no crowding distance, no knee point).
        """
        num_sentences = len(graph.nodes())
        if summary_size >= 1:
            target_length = max(1, min(int(summary_size), num_sentences))
        else:
            target_length = max(1, int(num_sentences * summary_size))

        node_importance = np.array([
            graph.nodes[n].get('importance', 0) for n in graph.nodes()
        ])
        if np.all(node_importance == 0):
            node_importance = np.array([
                graph.degree(n, weight='weight') for n in graph.nodes()
            ])
        adj_matrix = nx.to_numpy_array(graph, weight='weight')

        if position_bonus is None:
            position_bonus = np.ones(num_sentences)

        biased_importance = np.array([
            graph.nodes[i]['importance'] * position_bonus[i]
            for i in range(num_sentences)
        ])

        def evaluate_objectives(solution):
            selected = np.where(solution == 1)[0]
            coverage = biased_importance[selected].sum() if len(selected) > 0 else 0

            if len(selected) > 1:
                pairs = adj_matrix[np.ix_(selected, selected)]
                mask = pairs > 0.2
                np.fill_diagonal(mask, False)
                redundancy = pairs[mask].mean() if mask.any() else 0
            else:
                redundancy = 0

            length_penalty = abs(len(selected) - target_length)
            return coverage, redundancy, length_penalty

        def fitness(solution):
            cov, red, lpen = evaluate_objectives(solution)
            return cov - red - 2 * (lpen ** 2)

        def repair_solution(solution):
            sol = solution.copy()
            selected = np.where(sol == 1)[0]
            while len(selected) > target_length:
                importances = biased_importance[selected]
                drop_idx = selected[importances.argmin()]
                sol[drop_idx] = 0
                selected = np.where(sol == 1)[0]
            while len(selected) < target_length:
                unselected = np.where(sol == 0)[0]
                if len(unselected) == 0:
                    break
                importances = biased_importance[unselected]
                add_idx = unselected[importances.argmax()]
                sol[add_idx] = 1
                selected = np.where(sol == 1)[0]
            return sol

        def initialize_population():
            population = []
            probs = biased_importance / biased_importance.sum() \
                if biased_importance.sum() > 0 else np.ones(num_sentences) / num_sentences
            probs = np.clip(probs * 5, 0, 1)
            for _ in range(pop_size):
                sol = (np.random.random(num_sentences) < probs).astype(int)
                sol = repair_solution(sol)
                population.append(sol)
            return np.array(population)

        def sigmoid(x):
            return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

        def update_position_woa(current, leader, a, population):
            new_solution = current.copy()
            r1 = np.random.random()
            r2 = np.random.random()
            A = 2 * a * r1 - a
            C = 2 * r2
            p = np.random.random()
            b = 1

            if p < 0.5:
                if abs(A) < 1:
                    # encircling prey
                    D = np.abs(C * leader - current)
                    transfer = sigmoid(leader - A * D)
                    new_solution = (np.random.random(num_sentences) < transfer).astype(int)
                else:
                    # exploration, move towards a random whale
                    rand_idx = np.random.randint(0, pop_size)
                    rand_sol = population[rand_idx]
                    D = np.abs(C * rand_sol - current)
                    transfer = sigmoid(rand_sol - A * D)
                    new_solution = (np.random.random(num_sentences) < transfer).astype(int)
            else:
                # spiral bubble-net
                D_prime = np.abs(leader - current)
                l_param = np.random.uniform(-1, 1)
                spiral = D_prime * np.exp(b * l_param) * np.cos(2 * np.pi * l_param)
                transfer = sigmoid(leader + spiral)
                new_solution = (np.random.random(num_sentences) < transfer).astype(int)

            return repair_solution(new_solution)

        population = initialize_population()

        for t in range(max_iter):
            a = 2 - 2 * t / max_iter  # 2 -> 0

            # leader = best scalar fitness
            fitnesses = np.array([fitness(sol) for sol in population])
            leader_idx = fitnesses.argmax()
            leader = population[leader_idx].copy()

            for i in range(pop_size):
                population[i] = update_position_woa(
                    population[i], leader, a, population
                )

        fitnesses = np.array([fitness(sol) for sol in population])
        best_idx = fitnesses.argmax()
        best_solution = population[best_idx]

        return best_solution

    print(f"\nrunning single-obj WOA on {N_ARTICLES} articles (bge-m3 + WOA per article)\n")

    singleobj_woa_summaries = []
    start_time = time.time()

    for idx in range(N_ARTICLES):
        sentences = all_sentences[idx]
        n = len(sentences)

        # same position bonus as MO-WOA
        position_bonus = np.ones(n)
        if n > 0:
            position_bonus[0] = 3.0
        if n > 1:
            position_bonus[1:min(3, n)] *= 2.5
        if n > 3:
            position_bonus[3:min(5, n)] *= 1.5
        if n > 5:
            position_bonus[5:min(8, n)] *= 0.8
        if n > 8:
            position_bonus[8:] *= 0.3

        graph, _ = build_similarity_graph(sentences, bge_model, threshold=0.1)
        compute_lexrank(graph, damping=0.85)

        best_solution = single_objective_woa(
            graph, pop_size=30, max_iter=50,
            summary_size=SUMMARY_SIZE, position_bonus=position_bonus
        )

        selected_indices = np.where(best_solution == 1)[0]
        summary = ' '.join([sentences[i] for i in selected_indices])
        singleobj_woa_summaries.append(summary)

        if (idx + 1) % 10 == 0:
            elapsed = time.time() - start_time
            eta = elapsed / (idx + 1) * (N_ARTICLES - idx - 1)
            print(f"WOA {idx+1}/{N_ARTICLES}, "
                  f"{elapsed/60:.1f} min, eta {eta/60:.1f} min")

    total_time = time.time() - start_time
    print(f"\nsingle-obj WOA done in {total_time/60:.1f} min")

    singleobj_woa_results = evaluate_summaries(
        singleobj_woa_summaries, golden_summaries, source_texts, "WOA (Single-obj)"
    )

    # saved so next time it can just be loaded (RUN_SINGLE_OBJ_WOA = False)
    with open(WOA_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump({
            'generated_summaries': singleobj_woa_summaries,
        }, f, ensure_ascii=False, indent=2)
    print("saved", WOA_JSON_PATH)

else:
    try:
        with open(WOA_JSON_PATH, 'r', encoding='utf-8') as f:
            woa_data = json.load(f)
        singleobj_woa_summaries = woa_data['generated_summaries']
        print(f"loaded {len(singleobj_woa_summaries)} single-obj WOA summaries from {WOA_JSON_PATH}")

        singleobj_woa_results = evaluate_summaries(
            singleobj_woa_summaries, golden_summaries, source_texts, "WOA (Single-obj)"
        )
    except FileNotFoundError:
        print(f"{WOA_JSON_PATH} not found, set RUN_SINGLE_OBJ_WOA = True to compute it")
        singleobj_woa_summaries = None
        singleobj_woa_results = None

## 10. Comparison: IndoBERT (Fine-tuned Extractive Summarization)

IndoBERT (`indobenchmark/indobert-base-p2`) is a BERT model pre-trained on Indonesian text.
Fine-tuned for extractive summarization following BertSum (Liu & Lapata, 2019):

1. Extractive labels: for each article, greedily pick the source sentences that maximize
   ROUGE-L against the reference summary (oracle labels).
2. Fine-tune IndoBERT with a linear classification head to predict whether a sentence is included.
3. Inference: score every sentence in the 150 eval articles, take the top 3.

Training data: articles 150-2150 from `train.01.jsonl` (2000 articles, no overlap with the eval set).

Needs a GPU, takes ~30-60 minutes.

In [ ]:
# IndoBERT extractive (BertSum style). fine-tuned model and oracle labels are cached,
# so only the first run fine-tunes (~20-40 min), after that it's ~10-30s.
# put INDOBERT_MODEL_CACHE on drive so it survives colab restarts

import os
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

MODEL_NAME = 'indobenchmark/indobert-base-p2'
INDOBERT_MODEL_CACHE  = f'{DRIVE_CACHE_DIR}/indobert_extractor.pt'
INDOBERT_LABELS_CACHE = f'{DRIVE_CACHE_DIR}/indobert_oracle_labels.pkl'
INDOBERT_SUMMARIES_PATH = f'{DRIVE_CACHE_DIR}/indobert_summaries_150.json'

N_TRAIN_ARTICLES = 1000  # was 2000, barely any difference
EPOCHS           = 2     # was 3
BATCH_SIZE       = 64    # was 32
MAX_SENT_LEN     = 128
INFERENCE_BATCH  = 64

class IndoBERTExtractor(nn.Module):
    """IndoBERT + linear head on [CLS], one score per sentence."""
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = self.dropout(outputs.last_hidden_state[:, 0, :])
        return self.classifier(cls_output).squeeze(-1)

print("tokenizer:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.model_max_length > 1_000_000:
    tokenizer.model_max_length = 512

model = IndoBERTExtractor(MODEL_NAME).to(device)

if os.path.exists(INDOBERT_MODEL_CACHE):
    print(f"\nloading fine-tuned model from {INDOBERT_MODEL_CACHE}, skipping fine-tuning")
    model.load_state_dict(torch.load(INDOBERT_MODEL_CACHE, map_location=device))
else:
    print("\nno cached model, fine-tuning IndoBERT\n")

    # oracle labels
    if os.path.exists(INDOBERT_LABELS_CACHE):
        print("loading oracle labels from", INDOBERT_LABELS_CACHE)
        with open(INDOBERT_LABELS_CACHE, 'rb') as f:
            train_sentences_list, train_labels_list = pickle.load(f)
    else:
        def create_extractive_labels(sentences, reference_summary, n=3):
            """Greedy oracle: pick n sentences that maximize ROUGE-L F1."""
            scorer_local = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
            labels = np.zeros(len(sentences), dtype=int)
            selected, remaining = [], list(range(len(sentences)))
            for _ in range(min(n, len(sentences))):
                best_score, best_idx = -1, remaining[0] if remaining else 0
                for idx in remaining:
                    cand = ' '.join([sentences[s] for s in selected + [idx]])
                    s = scorer_local.score(reference_summary, cand)['rougeL'].fmeasure
                    if s > best_score:
                        best_score, best_idx = s, idx
                selected.append(best_idx)
                remaining.remove(best_idx)
            for idx in selected:
                labels[idx] = 1
            return labels

        # train on articles after the eval set so they never overlap
        TRAIN_START = max(N_ARTICLES, 150)
        TRAIN_END = min(TRAIN_START + N_TRAIN_ARTICLES, len(train_df))
        print(f"oracle labels for articles {TRAIN_START}..{TRAIN_END-1}")

        train_sentences_list, train_labels_list = [], []
        for idx in range(TRAIN_START, TRAIN_END):
            sents = split_text_into_sentences(train_df.paragraphs[idx])
            ref = tokens_to_narrative(train_df.summary[idx])
            if len(sents) < 2:
                continue
            labels = create_extractive_labels(sents, ref, n=SUMMARY_SIZE)
            train_sentences_list.append(sents)
            train_labels_list.append(labels)
            if (idx - TRAIN_START + 1) % 200 == 0:
                print(f"labels {idx - TRAIN_START + 1}/{TRAIN_END - TRAIN_START}")

        with open(INDOBERT_LABELS_CACHE, 'wb') as f:
            pickle.dump((train_sentences_list, train_labels_list), f)
        print("saved", INDOBERT_LABELS_CACHE)

    print(f"{len(train_sentences_list)} training articles")

    # one sample per sentence
    class SentenceDataset(Dataset):
        def __init__(self, sents_list, lbls_list, tok, max_length=128):
            self.data = []
            for sents, lbls in zip(sents_list, lbls_list):
                for sent, lbl in zip(sents, lbls):
                    enc = tok(sent, truncation=True, max_length=max_length,
                              padding='max_length', return_tensors='pt')
                    self.data.append({
                        'input_ids':      enc['input_ids'].squeeze(0),
                        'attention_mask': enc['attention_mask'].squeeze(0),
                        'label':          torch.tensor(lbl, dtype=torch.float),
                    })
        def __len__(self): return len(self.data)
        def __getitem__(self, idx): return self.data[idx]

    print("tokenizing...")
    train_dataset = SentenceDataset(train_sentences_list, train_labels_list,
                                    tokenizer, max_length=MAX_SENT_LEN)
    print(f"{len(train_dataset)} training sentences")

    n_pos = sum(1 for d in train_dataset.data if d['label'].item() == 1)
    n_neg = len(train_dataset) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], device=device)
    print(f"labels: {n_pos} pos / {n_neg} neg, pos_weight {pos_weight.item():.2f}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"\nfine-tuning, {EPOCHS} epochs, batch {BATCH_SIZE}")
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss, n_batches = 0.0, 0
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch['input_ids'].to(device),
                           batch['attention_mask'].to(device))
            loss = criterion(logits, batch['label'].to(device))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        print(f"epoch {epoch+1}/{EPOCHS} loss {epoch_loss/n_batches:.4f}")

    torch.save(model.state_dict(), INDOBERT_MODEL_CACHE)
    print("\nsaved", INDOBERT_MODEL_CACHE)
    del train_dataset, train_loader
    torch.cuda.empty_cache()

# inference, all sentences of an article in batches (much faster than one by one)
def score_sentences_batched(sentences, model, tok, batch_size=64, max_length=128):
    """Sigmoid score for every sentence."""
    if len(sentences) == 0:
        return []
    all_scores = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch_sents = sentences[i:i+batch_size]
            enc = tok(batch_sents, truncation=True, max_length=max_length,
                      padding=True, return_tensors='pt')
            logits = model(enc['input_ids'].to(device),
                           enc['attention_mask'].to(device))
            all_scores.extend(torch.sigmoid(logits).cpu().numpy().tolist())
    return all_scores

print(f"\ninference on {N_ARTICLES} articles")
indobert_summaries = []
inference_start = time.time()

for idx in range(N_ARTICLES):
    sentences = all_sentences[idx]
    if len(sentences) <= SUMMARY_SIZE:
        indobert_summaries.append(' '.join(sentences))
        continue
    scores = score_sentences_batched(sentences, model, tokenizer,
                                     batch_size=INFERENCE_BATCH, max_length=MAX_SENT_LEN)
    ranked = np.argsort(scores)[::-1]
    selected_indices = sorted(ranked[:SUMMARY_SIZE].tolist())
    indobert_summaries.append(' '.join([sentences[i] for i in selected_indices]))
    if (idx + 1) % 50 == 0:
        print(f"inference {idx+1}/{N_ARTICLES}, {time.time()-inference_start:.1f}s")

inf_time = time.time() - inference_start
print(f"\n{N_ARTICLES} articles in {inf_time:.1f}s "
      f"({inf_time/max(N_ARTICLES,1)*1000:.0f} ms/article)")

indobert_results = evaluate_summaries(
    indobert_summaries, golden_summaries, source_texts, "IndoBERT"
)

with open(INDOBERT_SUMMARIES_PATH, 'w', encoding='utf-8') as f:
    json.dump({'generated_summaries': indobert_summaries}, f,
              ensure_ascii=False, indent=2)
print("saved", INDOBERT_SUMMARIES_PATH)

torch.cuda.empty_cache()

## 11. Master Comparison Table

Consolidated results for all algorithms, evaluated on the same 150 IndoSum articles
with identical metrics, configuration, and canonical reference summaries.

In [ ]:
all_results = []
result_labels = []

for name, result in [
    ("LEAD-3", lead3_results),
    ("TextRank", textrank_results),
    ("LexRank", lexrank_results),
    ("WOA (Single-obj)", singleobj_woa_results if 'singleobj_woa_results' in dir() and singleobj_woa_results else None),
    ("IndoBERT", indobert_results if 'indobert_results' in dir() and indobert_results else None),
    ("MO-WOA", mowoa_results),
]:
    if result is not None:
        all_results.append(result)
        result_labels.append(name)
    else:
        print(f"{name} results missing, skipped")

table_data = []
for r in all_results:
    table_data.append({
        'Algorithm': r['algorithm'],
        'ROUGE-1 P': f"{r['rouge1_precision']:.4f}",
        'ROUGE-1 R': f"{r['rouge1_recall']:.4f}",
        'ROUGE-1 F1': f"{r['rouge1_f1']:.4f}",
        'ROUGE-2 P': f"{r['rouge2_precision']:.4f}",
        'ROUGE-2 R': f"{r['rouge2_recall']:.4f}",
        'ROUGE-2 F1': f"{r['rouge2_f1']:.4f}",
        'ROUGE-L P': f"{r['rougeL_precision']:.4f}",
        'ROUGE-L R': f"{r['rougeL_recall']:.4f}",
        'ROUGE-L F1': f"{r['rougeL_f1']:.4f}",
        'CR': f"{r['compression_ratio']:.1%}",
        'Avg Len': f"{r['avg_summary_length']:.0f}",
        'Redundancy': f"{r['ngram_redundancy']:.4f}",
    })

comparison_df = pd.DataFrame(table_data)

print("\nall algorithms, IndoSum 150 articles, 3 sentences, use_stemmer=False, trigram redundancy threshold 0.3")
print(comparison_df.to_string(index=False))

print("\nF1 only:")
f1_data = []
for r in all_results:
    f1_data.append({
        'Algorithm': r['algorithm'],
        'ROUGE-1 F1': f"{r['rouge1_f1']:.4f}",
        'ROUGE-2 F1': f"{r['rouge2_f1']:.4f}",
        'ROUGE-L F1': f"{r['rougeL_f1']:.4f}",
    })
f1_df = pd.DataFrame(f1_data)
print(f1_df.to_string(index=False))

if mowoa_results is not None:
    print("\nF1 difference vs MO-WOA (positive = MO-WOA better)")
    for r in all_results:
        if r['algorithm'] != 'MO-WOA':
            d1 = mowoa_results['rouge1_f1'] - r['rouge1_f1']
            d2 = mowoa_results['rouge2_f1'] - r['rouge2_f1']
            dL = mowoa_results['rougeL_f1'] - r['rougeL_f1']
            sign1 = '+' if d1 >= 0 else ''
            sign2 = '+' if d2 >= 0 else ''
            signL = '+' if dL >= 0 else ''
            print(f"  vs {r['algorithm']:<20s}  "
                  f"R1 {sign1}{d1:.4f}  R2 {sign2}{d2:.4f}  RL {signL}{dL:.4f}")

## 12. Export Results

Export the comparison table as TSV, so it can go into Word with Insert > Table > Convert Text to Table.

In [ ]:
tsv_filename = f'{DRIVE_CACHE_DIR}/mowoa_comparison_results.tsv'
comparison_df.to_csv(tsv_filename, sep='\t', index=False)
print("saved", tsv_filename)

# same table printed, to paste into word
print("\n(tab-separated)")
print('\t'.join(comparison_df.columns))
for _, row in comparison_df.iterrows():
    print('\t'.join(str(v) for v in row.values))

# per-article F1 for the significance tests (paired t-test, wilcoxon)
per_article_data = {}
for r in all_results:
    algo = r['algorithm']
    per_article_data[f'{algo}_R1_F1'] = r['_rouge1_f1_per_article']
    per_article_data[f'{algo}_R2_F1'] = r['_rouge2_f1_per_article']
    per_article_data[f'{algo}_RL_F1'] = r['_rougeL_f1_per_article']

per_article_df = pd.DataFrame(per_article_data)
per_article_filename = f'{DRIVE_CACHE_DIR}/mowoa_per_article_f1_scores.tsv'
per_article_df.to_csv(per_article_filename, sep='\t', index=True)
print("\nsaved", per_article_filename)

### 12B More Complete ROUGE Result Table

In [ ]:
# tabel 4.18 diperluas (rata-rata, SD, min, maks per artikel) + contoh ringkasan terbaik MOWOA
MOWOA_W32_JSON  = 'mowoa_summaries_150_rc1_w32.json'  # sumber baris MOWOA Tabel 4.18
PER_ARTICLE_TSV = f'{DRIVE_CACHE_DIR}/mowoa_per_article_f1_scores.tsv'

THESIS_MEANS = {   # nilai Tabel 4.18 saat ini, untuk verifikasi
    'MOWOA (proposed)':          (0.6544, 0.5852, 0.6168),
    'LEAD-3':                    (0.6484, 0.5790, 0.6110),
    'WOA Single-obj (ablation)': (0.6141, 0.5381, 0.5756),
    'IndoBERT':                  (0.4766, 0.3795, 0.4303),
    'TextRank':                  (0.4229, 0.2856, 0.3427),
    'LexRank':                   (0.4211, 0.2838, 0.3421),
}
TSV_PREFIX = {'LEAD-3': 'LEAD-3', 'WOA Single-obj (ablation)': 'WOA (Single-obj)',
              'IndoBERT': 'IndoBERT', 'TextRank': 'TextRank', 'LexRank': 'LexRank'}
METRICS = [('ROUGE-1 F1', 'R1_F1'), ('ROUGE-2 F1', 'R2_F1'), ('ROUGE-L F1', 'RL_F1')]

# skor per artikel
per_article_df = pd.read_csv(PER_ARTICLE_TSV, sep='\t', index_col=0)
with open(MOWOA_W32_JSON, encoding='utf-8') as f:
    w32 = json.load(f)
assert w32['golden_summaries'][:N_ARTICLES] == golden_summaries, "Referensi berbeda!"

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
mowoa_w32 = w32['generated_summaries'][:N_ARTICLES]
mowoa_scores = np.array([[scorer.score(ref, gen)[m].fmeasure for m in ('rouge1', 'rouge2', 'rougeL')]
                         for gen, ref in zip(mowoa_w32, golden_summaries)])

scores = {'MOWOA (proposed)': mowoa_scores}
for name, prefix in TSV_PREFIX.items():
    scores[name] = per_article_df[[f'{prefix}_{c}' for _, c in METRICS]].to_numpy()

# verifikasi rata-rata terhadap Tabel 4.18
for name, arr in scores.items():
    got = tuple(np.round(arr.mean(axis=0), 4))
    ok = np.allclose(got, THESIS_MEANS[name], atol=5e-5)
    print(f"{name:<28} rata-rata {got}  vs Tabel 4.18 {THESIS_MEANS[name]}  -> "
          f"{'OK' if ok else 'MISMATCH (data per-artikel bukan dari run tesis)'}")

# tabel diperluas, blok baris per metrik, TSV untuk Word
fmt = lambda v: f'{v:.4f}'.replace('.', ',')
print('\nMetrik\tAlgoritma\tRata-rata\tSimpangan baku\tMin\tMaks')
for k, (label, _) in enumerate(METRICS):
    for name, arr in scores.items():
        x = arr[:, k]
        print(f"{label}\t{name}\t{fmt(x.mean())}\t{fmt(x.std(ddof=1))}\t{fmt(x.min())}\t{fmt(x.max())}")

# contoh kasus terbaik MOWOA
best = int(np.argmax(mowoa_scores.mean(axis=1)))
print(f"\nKriteria: rata-rata ROUGE-1/2/L F1 tertinggi -> Artikel {best + 1}")
for k, (label, _) in enumerate(METRICS):
    print(f"  artikel dengan {label} tertinggi: Artikel {int(np.argmax(mowoa_scores[:, k])) + 1}")
lead3_best = ' '.join(split_canonical_source(source_texts[best])[:SUMMARY_SIZE])
print("\nRingkasan acuan :", golden_summaries[best])
print("\nRingkasan MOWOA :", mowoa_w32[best])
print("\nRingkasan LEAD-3:", lead3_best)
print("\nSkor MOWOA  (R-1, R-2, R-L):", np.round(mowoa_scores[best], 4))
print("Skor LEAD-3 (R-1, R-2, R-L):", np.round(scores['LEAD-3'][best], 4))
print("MOWOA identik dengan LEAD-3:", mowoa_w32[best].strip() == lead3_best.strip())

## 13. Save All Generated Summaries

Archive all generated summaries for reproducibility and further analysis.

In [ ]:
all_summaries_archive = {
    'metadata': {
        'n_articles': N_ARTICLES,
        'summary_size': SUMMARY_SIZE,
        'dataset': 'IndoSum train.01.jsonl (first 150 articles)',
        'rouge_config': 'use_stemmer=False',
        'canonical_reference': 'mowoa_summaries_150.json',
    },
    'golden_summaries': golden_summaries,
    'source_texts': source_texts,
    'algorithms': {}
}

for name, summaries_var in [
    ('LEAD-3', 'lead3_summaries'),
    ('TextRank', 'textrank_summaries'),
    ('LexRank', 'lexrank_summaries'),
    ('WOA_SingleObj', 'singleobj_woa_summaries'),
    ('IndoBERT', 'indobert_summaries'),
    ('MOWOA', 'mowoa_summaries'),
]:
    if summaries_var in dir() and eval(summaries_var) is not None:
        all_summaries_archive['algorithms'][name] = eval(summaries_var)

archive_filename = f'{DRIVE_CACHE_DIR}/all_summaries_150_archive.json'
with open(archive_filename, 'w', encoding='utf-8') as f:
    json.dump(all_summaries_archive, f, ensure_ascii=False, indent=2)
print("saved", archive_filename)
print("algorithms:", list(all_summaries_archive['algorithms'].keys()))